## GridLock 2.0 - Travel Demand Forecast v2 (target 93+)

Improvements over v1 (91.44):
1. Multi-day SVD prior (day-48 + day-49 early actuals blended, 16 components)
2. Today-deviation feature per geohash
3. Fixed lag features for test (uses day-49 actuals, not stale day-48 tail)
4. Time-bucket-specific neighbour demand
5. Demand-profile clustering (shape-based, 30 clusters) + spatial (40 clusters)
6. OOF-optimised blend weights
7. Per-geohash level calibration from day-49 early actuals
8. Wider rolling windows, more interactions, lag_4, diff_2

In [ ]:
!pip install -q pygeohash catboost

In [ ]:
import warnings
import numpy as np
import pandas as pd
import pygeohash as pgh
from scipy.optimize import minimize
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import HistGradientBoostingRegressor, ExtraTreesRegressor
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)

TRAIN_PATH = '/content/train.csv'
TEST_PATH  = '/content/test.csv'

def add_time(df):
    parts = df['timestamp'].str.split(':', expand=True).astype(int)
    df['hour']        = parts[0]
    df['minute']      = parts[1]
    df['time_bucket'] = df['hour'] * 4 + df['minute'] // 15
    return df

train_raw = add_time(pd.read_csv(TRAIN_PATH))
test_raw  = add_time(pd.read_csv(TEST_PATH))
print('train:', train_raw.shape, ' days:', sorted(train_raw.day.unique()))
print('test :', test_raw.shape)

In [ ]:
# ── 1. GEOSPATIAL SETUP ──────────────────────────────────────────
cells = list(set(train_raw['geohash']) | set(test_raw['geohash']))
coord = {g: (pgh.decode(g).latitude, pgh.decode(g).longitude) for g in cells}

STEPS = ['top','bottom','left','right','topleft','topright','bottomleft','bottomright']
known = set(cells)
adj = {}
for g in cells:
    try:
        adj[g] = [pgh.get_adjacent(g, s) for s in STEPS if pgh.get_adjacent(g, s) in known]
    except Exception:
        adj[g] = []
print(f'Cells: {len(cells)}')

In [ ]:
# ── 2. HISTORICAL STATISTICS ─────────────────────────────────────
hist48 = train_raw[train_raw['day'] == 48].copy()
hist49 = train_raw[train_raw['day'] == 49].copy()
print(f'Day48 rows:{len(hist48)}  Day49 rows:{len(hist49)}')
print(f'Day49 buckets: {sorted(hist49.time_bucket.unique())}')

bucket_d48  = hist48.groupby(['geohash','time_bucket'])['demand'].mean().to_dict()
hour_d48    = hist48.groupby(['geohash','hour'])['demand'].mean().to_dict()
g_mean48    = hist48.groupby('geohash')['demand'].mean().to_dict()
g_std48     = hist48.groupby('geohash')['demand'].std().fillna(0).to_dict()
g_max48     = hist48.groupby('geohash')['demand'].max().to_dict()
g_min48     = hist48.groupby('geohash')['demand'].min().to_dict()
g_med48     = hist48.groupby('geohash')['demand'].median().to_dict()
hourly48    = hist48.groupby('hour')['demand'].mean().to_dict()
bucketly48  = hist48.groupby('time_bucket')['demand'].mean().to_dict()

bucket_d49  = hist49.groupby(['geohash','time_bucket'])['demand'].mean().to_dict()

glob_mean   = hist48['demand'].mean()
temp_fill   = hist48['Temperature'].median()
print(f'Global mean: {glob_mean:.4f}')

In [ ]:
# ── 3. SVD-SMOOTHED MULTI-DAY PRIOR (16 components) ─────────────
idx_map = {g: i for i, g in enumerate(cells)}
N = len(cells)

grid48 = np.full((N, 96), np.nan)
for (g, b), v in bucket_d48.items():
    if g in idx_map:
        grid48[idx_map[g], b] = v

grid49 = np.full((N, 96), np.nan)
for (g, b), v in bucket_d49.items():
    if g in idx_map:
        grid49[idx_map[g], b] = v

# where day-49 observed: 60% day-49 + 40% day-48; else day-48
grid_comb  = np.where(~np.isnan(grid49), 0.6*grid49 + 0.4*grid48, grid48)
base_level = np.nan_to_num(np.nanmean(grid_comb, axis=1), nan=glob_mean)
centred    = np.where(np.isnan(grid_comb), base_level[:,None], grid_comb) - base_level[:,None]

svd = TruncatedSVD(n_components=16, random_state=SEED)
smooth_grid = svd.inverse_transform(svd.fit_transform(centred)) + base_level[:,None]

def profile(g, b):
    if g in idx_map and 0 <= b < 96:
        return float(smooth_grid[idx_map[g], b])
    return g_mean48.get(g, glob_mean)

print(f'SVD explained variance: {svd.explained_variance_ratio_.sum():.4f}')

In [ ]:
# ── 4. CLUSTERING (spatial + demand-profile) ─────────────────────
coord_arr = np.array([coord[g] for g in cells])
km_spatial  = KMeans(n_clusters=40, random_state=SEED, n_init=10)
spatial_cluster = dict(zip(cells, km_spatial.fit_predict(coord_arr)))

prof_arr  = smooth_grid.copy()
row_std   = prof_arr.std(axis=1, keepdims=True)
row_std[row_std == 0] = 1.0
prof_norm = (prof_arr - prof_arr.mean(axis=1, keepdims=True)) / row_std
km_profile = KMeans(n_clusters=30, random_state=SEED, n_init=10)
profile_cluster = dict(zip(cells, km_profile.fit_predict(prof_norm)))

h48tmp = hist48.copy()
h48tmp['sp_cl'] = h48tmp['geohash'].map(spatial_cluster)
h48tmp['pr_cl'] = h48tmp['geohash'].map(profile_cluster)
sp_cl_demand = h48tmp.groupby('sp_cl')['demand'].mean().to_dict()
pr_cl_demand = h48tmp.groupby('pr_cl')['demand'].mean().to_dict()

# time-bucket-specific neighbour demand
nbr_bkt_demand = {}
for g in cells:
    ns = adj.get(g, [])
    for b in range(96):
        vals = [v for n in ns for v in [bucket_d48.get((n,b))] if v is not None and not np.isnan(v)]
        nbr_bkt_demand[(g,b)] = float(np.mean(vals)) if vals else bucket_d48.get((g,b), g_mean48.get(g, glob_mean))

adj_demand = {g: np.mean([g_mean48.get(n,0) for n in ns]) if ns else g_mean48.get(g,0)
              for g, ns in adj.items()}
print('Clustering done.')

In [ ]:
# ── 5. TODAY-DEVIATION FEATURE ───────────────────────────────────
today_buckets = sorted(hist49['time_bucket'].unique())
today_dev = {}
for g in cells:
    obs  = np.array([bucket_d49.get((g,b), np.nan) for b in today_buckets])
    prof = np.array([profile(g, b)                  for b in today_buckets])
    mask = ~np.isnan(obs)
    if mask.sum() > 0 and np.mean(prof[mask]) > 1e-6:
        today_dev[g] = float(np.mean(obs[mask]) / np.mean(prof[mask]))
    else:
        today_dev[g] = 1.0

tdv = list(today_dev.values())
print(f'today_dev  min={min(tdv):.3f}  mean={np.mean(tdv):.3f}  max={max(tdv):.3f}')

In [ ]:
# ── 6. FEATURE ENGINEERING ───────────────────────────────────────
curve48 = {g: np.array([bucket_d48.get((g,b), np.nan) for b in range(96)]) for g in cells}

def around(g, b, w, fn):
    seg = curve48[g][max(0, b-w): b+w+1]
    seg = seg[~np.isnan(seg)]
    return float(fn(seg)) if len(seg) else g_mean48.get(g, glob_mean)

def build_features(df):
    df = df.copy()
    df['lat']         = df['geohash'].map(lambda x: coord.get(x,(0,0))[0])
    df['lon']         = df['geohash'].map(lambda x: coord.get(x,(0,0))[1])
    df['sp_cluster']  = df['geohash'].map(spatial_cluster).fillna(0).astype(int)
    df['pr_cluster']  = df['geohash'].map(profile_cluster).fillna(0).astype(int)
    df['dow']         = (df['day'] - 1) % 7
    df['is_peak']     = (((df['hour']>=7)&(df['hour']<=9))|((df['hour']>=17)&(df['hour']<=19))).astype(int)
    df['is_night']    = ((df['hour']>=22)|(df['hour']<=5)).astype(int)
    df['is_lunch']    = ((df['hour']>=11)&(df['hour']<=13)).astype(int)
    df['hour_sin']    = np.sin(2*np.pi*df['hour']/24)
    df['hour_cos']    = np.cos(2*np.pi*df['hour']/24)
    df['bkt_sin']     = np.sin(2*np.pi*df['time_bucket']/96)
    df['bkt_cos']     = np.cos(2*np.pi*df['time_bucket']/96)
    df['bkt_sin2']    = np.sin(4*np.pi*df['time_bucket']/96)
    df['bkt_cos2']    = np.cos(4*np.pi*df['time_bucket']/96)
    df['bkt_sin3']    = np.sin(6*np.pi*df['time_bucket']/96)
    df['bkt_cos3']    = np.cos(6*np.pi*df['time_bucket']/96)
    df['road']        = df['RoadType'].map({'Residential':0,'Street':1,'Highway':2}).fillna(-1)
    df['large']       = (df['LargeVehicles']=='Allowed').astype(int)
    df['landmark']    = (df['Landmarks']=='Yes').astype(int)
    df['weather']     = df['Weather'].map({'Sunny':0,'Rainy':1,'Foggy':2,'Snowy':3}).fillna(-1)
    df['temp']        = df['Temperature'].fillna(temp_fill)
    df['temp_weather']= df['temp']*df['weather']
    df['temp_sq']     = df['temp']**2
    df['cold_flag']   = (df['temp']<10).astype(int)
    df['lanes_road']  = df['NumberofLanes']*df['road']
    df['lmk_peak']    = df['landmark']*df['is_peak']
    df['hw_peak']     = (df['road']==2).astype(int)*df['is_peak']
    for off in (-2,-1,0,1,2):
        df[f'd48_b{off}'] = df.apply(
            lambda r: bucket_d48.get((r['geohash'], r['time_bucket']+off),
                                     g_mean48.get(r['geohash'], glob_mean)), axis=1)
    df['d48_w3m']  = df.apply(lambda r: around(r['geohash'],r['time_bucket'],3,np.mean),axis=1)
    df['d48_w3s']  = df.apply(lambda r: around(r['geohash'],r['time_bucket'],3,np.std), axis=1)
    df['d48_w6m']  = df.apply(lambda r: around(r['geohash'],r['time_bucket'],6,np.mean),axis=1)
    df['d48_hour'] = df.apply(lambda r: hour_d48.get((r['geohash'],r['hour']),g_mean48.get(r['geohash'],glob_mean)),axis=1)
    df['geo_mean'] = df['geohash'].map(g_mean48).fillna(glob_mean)
    df['geo_std']  = df['geohash'].map(g_std48).fillna(0)
    df['geo_max']  = df['geohash'].map(g_max48).fillna(0)
    df['geo_min']  = df['geohash'].map(g_min48).fillna(0)
    df['geo_med']  = df['geohash'].map(g_med48).fillna(0)
    df['hour_glob']= df['hour'].map(hourly48).fillna(glob_mean)
    df['bkt_glob'] = df['time_bucket'].map(bucketly48).fillna(glob_mean)
    df['sp_cl_mean']= df['sp_cluster'].map(sp_cl_demand).fillna(glob_mean)
    df['pr_cl_mean']= df['pr_cluster'].map(pr_cl_demand).fillna(glob_mean)
    df['nbr_mean'] = df['geohash'].map(adj_demand).fillna(glob_mean)
    df['nbr_bkt_mean'] = df.apply(
        lambda r: nbr_bkt_demand.get((r['geohash'],r['time_bucket']),
                                     adj_demand.get(r['geohash'],glob_mean)), axis=1)
    df['d48b0_peak']   = df['d48_b0']*df['is_peak']
    df['d48b0_vs_geo'] = df['d48_b0']/(df['geo_mean']+1e-9)
    df['nbr_vs_geo']   = df['nbr_bkt_mean']/(df['geo_mean']+1e-9)
    df['prof0']   = [profile(g,b)   for g,b in zip(df['geohash'],df['time_bucket'])]
    df['prof_m1'] = [profile(g,b-1) for g,b in zip(df['geohash'],df['time_bucket'])]
    df['prof_p1'] = [profile(g,b+1) for g,b in zip(df['geohash'],df['time_bucket'])]
    df['prof_m2'] = [profile(g,b-2) for g,b in zip(df['geohash'],df['time_bucket'])]
    df['prof_p2'] = [profile(g,b+2) for g,b in zip(df['geohash'],df['time_bucket'])]
    df['prof_slope'] = df['prof_p1'] - df['prof_m1']
    df['today_dev']  = df['geohash'].map(today_dev).fillna(1.0)
    df['adj_prof0']  = df['prof0'] * df['today_dev']
    return df

print('Building train features...')
train_feat = build_features(train_raw)
print('Building test features...')
test_feat  = build_features(test_raw)
print('Done.')

In [ ]:
# ── 7. LAG FEATURES (FIXED for test) ────────────────────────────
train_feat = train_feat.sort_values(['geohash','day','time_bucket']).reset_index(drop=True)
for i in (1,2,3,4):
    train_feat[f'lag_{i}'] = train_feat.groupby('geohash')['demand'].shift(i)
train_feat['diff_1'] = train_feat['lag_1'] - train_feat['lag_2']
train_feat['diff_2'] = train_feat['lag_2'] - train_feat['lag_3']

def get_lag(g, bucket, k):
    src = bucket - k
    if 0 <= src <= 8:    # day-49 known actuals
        return bucket_d49.get((g,src), bucket_d48.get((g,src), g_mean48.get(g, glob_mean)))
    elif src < 0:        # wrap to end of day-48
        return bucket_d48.get((g, 96+src), g_mean48.get(g, glob_mean))
    else:                # day-48 bucket
        return bucket_d48.get((g,src), g_mean48.get(g, glob_mean))

for k in (1,2,3,4):
    test_feat[f'lag_{k}'] = [get_lag(g,b,k) for g,b in zip(test_feat['geohash'],test_feat['time_bucket'])]
test_feat['diff_1'] = test_feat['lag_1'] - test_feat['lag_2']
test_feat['diff_2'] = test_feat['lag_2'] - test_feat['lag_3']
print('Lags done.')

In [ ]:
# ── 8. FEATURE MATRIX ────────────────────────────────────────────
COLS = [
    'lat','lon','sp_cluster','pr_cluster',
    'hour','minute','time_bucket','dow',
    'is_peak','is_night','is_lunch',
    'hour_sin','hour_cos',
    'bkt_sin','bkt_cos','bkt_sin2','bkt_cos2','bkt_sin3','bkt_cos3',
    'road','NumberofLanes','large','landmark',
    'lanes_road','lmk_peak','hw_peak',
    'weather','temp','temp_weather','temp_sq','cold_flag',
    'd48_b-2','d48_b-1','d48_b0','d48_b1','d48_b2',
    'd48_w3m','d48_w3s','d48_w6m','d48_hour',
    'geo_mean','geo_std','geo_max','geo_min','geo_med',
    'hour_glob','bkt_glob',
    'sp_cl_mean','pr_cl_mean',
    'nbr_mean','nbr_bkt_mean',
    'd48b0_peak','d48b0_vs_geo','nbr_vs_geo',
    'prof0','prof_m1','prof_p1','prof_m2','prof_p2','prof_slope',
    'today_dev','adj_prof0',
    'lag_1','lag_2','lag_3','lag_4','diff_1','diff_2',
]

data  = train_feat.dropna(subset=['lag_1']).copy()
prior = data['adj_prof0'].values
y     = data['demand'].values - prior

X     = data[COLS].astype(np.float32).replace([np.inf,-np.inf],0).fillna(0)
X_tst = test_feat[COLS].astype(np.float32).replace([np.inf,-np.inf],0).fillna(0)
prior_tst = test_feat['adj_prof0'].values

# day-48 / day-49 splits for early-stopping and OOF
fit_mask  = data['day'] == 48
hold_mask = data['day'] == 49
X_fit  = data.loc[fit_mask,  COLS].astype(np.float32).fillna(0)
y_fit  = (data.loc[fit_mask,  'demand'] - data.loc[fit_mask,  'adj_prof0']).values
X_hold = data.loc[hold_mask, COLS].astype(np.float32).fillna(0)
y_hold = (data.loc[hold_mask, 'demand'] - data.loc[hold_mask, 'adj_prof0']).values

print(f'Train: {X.shape}  Test: {X_tst.shape}')
print(f'Fit fold: {X_fit.shape}  Hold fold: {X_hold.shape}')

In [ ]:
# ── 9. EARLY STOPPING TO PICK n_trees ───────────────────────────
probe = lgb.LGBMRegressor(
    objective='regression', metric='rmse',
    learning_rate=0.025, num_leaves=255,
    min_child_samples=20, feature_fraction=0.75,
    bagging_fraction=0.8, bagging_freq=5,
    reg_alpha=0.1, reg_lambda=0.5,
    n_estimators=3000, verbose=-1,
    random_state=SEED, n_jobs=-1
)
probe.fit(
    X_fit, y_fit,
    eval_set=[(X_hold, y_hold)],
    callbacks=[lgb.early_stopping(80, verbose=False)]
)
n_trees = max(150, probe.best_iteration_ or 300)
print(f'n_trees = {n_trees}')

In [ ]:
# ── 10. OOF PREDICTIONS FOR WEIGHT OPTIMISATION ─────────────────
# Each model trained on day-48 only, predicts on day-49 early hold-out
print('OOF: LGB...')
lgb_oof_runs = []
for sd in (42, 7, 2024, 137):
    m = lgb.LGBMRegressor(
        objective='regression', metric='rmse',
        learning_rate=0.025, num_leaves=255,
        min_child_samples=20, feature_fraction=0.75,
        bagging_fraction=0.8, bagging_freq=5,
        reg_alpha=0.1, reg_lambda=0.5,
        n_estimators=n_trees, verbose=-1, random_state=sd, n_jobs=-1)
    m.fit(X_fit, y_fit)
    lgb_oof_runs.append(m.predict(X_hold))
lgb_oof_model = m   # keep last for calib
oof_lgb = np.mean(lgb_oof_runs, axis=0)

print('OOF: XGB...')
xgb_oof = xgb.XGBRegressor(
    objective='reg:squarederror', learning_rate=0.025,
    max_depth=7, min_child_weight=40,
    subsample=0.8, colsample_bytree=0.75,
    reg_alpha=0.1, reg_lambda=1.0,
    n_estimators=n_trees, random_state=SEED,
    tree_method='hist', n_jobs=-1, verbosity=0)
xgb_oof.fit(X_fit, y_fit)
oof_xgb = xgb_oof.predict(X_hold)

print('OOF: CatBoost...')
cat_oof = CatBoostRegressor(
    iterations=n_trees, learning_rate=0.025,
    depth=7, l2_leaf_reg=3,
    random_seed=SEED, verbose=0, task_type='CPU')
cat_oof.fit(X_fit, y_fit)
oof_cat = cat_oof.predict(X_hold)

print('OOF: HistGBM...')
hgb_oof = HistGradientBoostingRegressor(
    max_iter=min(n_trees,500), learning_rate=0.025,
    max_leaf_nodes=127, l2_regularization=0.5, random_state=SEED)
hgb_oof.fit(X_fit, y_fit)
oof_hgb = hgb_oof.predict(X_hold)

print('OOF: ExtraTrees...')
ext_oof = ExtraTreesRegressor(
    n_estimators=400, max_features=0.6,
    min_samples_leaf=15, random_state=SEED, n_jobs=-1)
ext_oof.fit(X_fit, y_fit)
oof_ext = ext_oof.predict(X_hold)

# Optimise weights
oof_mat = np.stack([oof_lgb, oof_xgb, oof_cat, oof_hgb, oof_ext], axis=1)

def neg_rmse(w):
    w = np.clip(w, 0, 1); w /= (w.sum() + 1e-12)
    return float(np.sqrt(np.mean((y_hold - oof_mat @ w)**2)))

res = minimize(neg_rmse, [0.45,0.20,0.13,0.12,0.10],
               method='Nelder-Mead',
               options={'maxiter':5000,'xatol':1e-6,'fatol':1e-6})
opt_w = np.clip(res.x, 0, 1); opt_w /= opt_w.sum()
print(f'Optimal weights: LGB={opt_w[0]:.3f} XGB={opt_w[1]:.3f} '
      f'CAT={opt_w[2]:.3f} HGB={opt_w[3]:.3f} EXT={opt_w[4]:.3f}')
for lbl, w in [('v1 hand', [0.45,0.20,0.13,0.12,0.10]),('optimised', opt_w)]:
    ww = np.array(w); ww /= ww.sum()
    print(f'  OOF RMSE [{lbl}]: {np.sqrt(np.mean((y_hold - oof_mat@ww)**2)):.6f}')

In [ ]:
# ── 11. TRAIN FINAL MODELS ON FULL TRAINING DATA ─────────────────
print('Final: LGB...')
lgb_preds = []
for sd in (42, 7, 2024, 137):
    m = lgb.LGBMRegressor(
        objective='regression', metric='rmse',
        learning_rate=0.025, num_leaves=255,
        min_child_samples=20, feature_fraction=0.75,
        bagging_fraction=0.8, bagging_freq=5,
        reg_alpha=0.1, reg_lambda=0.5,
        n_estimators=n_trees, verbose=-1, random_state=sd, n_jobs=-1)
    m.fit(X, y)
    lgb_preds.append(m.predict(X_tst))
p_lgb = np.mean(lgb_preds, axis=0)

print('Final: XGB...')
xgb_m = xgb.XGBRegressor(
    objective='reg:squarederror', learning_rate=0.025,
    max_depth=7, min_child_weight=40,
    subsample=0.8, colsample_bytree=0.75,
    reg_alpha=0.1, reg_lambda=1.0,
    n_estimators=n_trees, random_state=SEED,
    tree_method='hist', n_jobs=-1, verbosity=0)
xgb_m.fit(X, y)
p_xgb = xgb_m.predict(X_tst)

print('Final: CatBoost...')
cat_m = CatBoostRegressor(
    iterations=n_trees, learning_rate=0.025,
    depth=7, l2_leaf_reg=3,
    random_seed=SEED, verbose=0, task_type='CPU')
cat_m.fit(X, y)
p_cat = cat_m.predict(X_tst)

print('Final: HistGBM...')
hgb = HistGradientBoostingRegressor(
    max_iter=min(n_trees,500), learning_rate=0.025,
    max_leaf_nodes=127, l2_regularization=0.5, random_state=SEED)
hgb.fit(X, y)
p_hgb = hgb.predict(X_tst)

print('Final: ExtraTrees...')
ext = ExtraTreesRegressor(
    n_estimators=400, max_features=0.6,
    min_samples_leaf=15, random_state=SEED, n_jobs=-1)
ext.fit(X, y)
p_ext = ext.predict(X_tst)

print('All models trained.')

In [ ]:
# ── 12. BLEND + LEVEL CALIBRATION ────────────────────────────────
# Blend test predictions
all_test = np.stack([p_lgb, p_xgb, p_cat, p_hgb, p_ext], axis=1)
raw_resid = all_test @ opt_w
raw_pred  = np.clip(prior_tst + raw_resid, 0, 1)

# ── per-geohash level calibration ────────────────────────────────
# Use OOF models (trained on day-48) to predict on day-49 early rows,
# then compare to actuals to build a correction ratio per geohash.
# All OOF models are already fitted — no new fitting needed here.

calib_data  = data[data['day'] == 49].copy()
calib_X     = calib_data[COLS].astype(np.float32).replace([np.inf,-np.inf],0).fillna(0)
calib_prior = calib_data['adj_prof0'].values

# OOF predictions on calib set (reuse already-fitted OOF models)
calib_lgb_preds = np.mean([m.predict(calib_X) for m in [
    # reuse lgb_oof_model (last seed fitted on day-48)
    lgb_oof_model
]], axis=0)
calib_xgb = xgb_oof.predict(calib_X)
calib_cat = cat_oof.predict(calib_X)
calib_hgb = hgb_oof.predict(calib_X)
calib_ext = ext_oof.predict(calib_X)

calib_resid_blend = (np.stack([calib_lgb_preds, calib_xgb, calib_cat, calib_hgb, calib_ext], axis=1) @ opt_w)
calib_pred  = np.clip(calib_prior + calib_resid_blend, 0, 1)
calib_actual = calib_data['demand'].values

# per-geohash ratio
calib_df = pd.DataFrame({
    'geohash': calib_data['geohash'].values,
    'actual':  calib_actual,
    'pred':    calib_pred
})
geo_ratio = (
    calib_df.groupby('geohash')
    .apply(lambda d: d['actual'].mean() / (d['pred'].mean() + 1e-9))
    .clip(0.5, 2.0)
    .to_dict()
)
global_ratio = np.clip(calib_actual.mean() / (calib_pred.mean() + 1e-9), 0.7, 1.4)

test_ratios = test_feat['geohash'].map(lambda g: geo_ratio.get(g, global_ratio)).values
calibrated_pred = np.clip(raw_pred * test_ratios, 0, 1)

print(f'Global level ratio: {global_ratio:.4f}')
print(f'Before calib mean: {raw_pred.mean():.4f}  After calib mean: {calibrated_pred.mean():.4f}')

In [ ]:
# ── 13. WRITE SUBMISSION ─────────────────────────────────────────
out = pd.DataFrame({'Index': test_feat['Index'].values, 'demand': calibrated_pred})
out.to_csv('submission.csv', index=False)
print('submission.csv written:', out.shape)
print(f'demand  min={calibrated_pred.min():.4f}  mean={calibrated_pred.mean():.4f}  max={calibrated_pred.max():.4f}')
out.head(10)